## YOLO V10 video detection

本文件展示如何使用YOLO v10完成师表目标检测

实验流程
- 实验环境配置
- 安装YOLOv10
- 加载预训练模型
- 输入测试视频
- 执行目标加测
- 展示检测结果

### 1、实验环境配置
#### 1.1、检查Python版本和路径


In [1]:
!Python --version

Python 3.12.8


In [2]:
import sys

print("当前python路径:", sys.executable)

当前python路径: c:\Python\Python312\python.exe


#### 1.2 安装PyTorch

Pytorch 是YOLO V10 运行的核心深度学习框架

In [3]:
%pip install -U pip
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126


  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Looking in indexes: https://download.pytorch.org/whl/cu126
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch

print("torch:", torch.__version__)
print("torch cuda build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.9.1+cu126
torch cuda build: 12.6
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


#### 1.3、安装实验依赖库

YOLOv10 运行需要多个Python第三方库，主要包括：
 - 'Ultralytics': 提供YOLO模型调用接口 
 - OpenCV: 负责视频读取和处理
 - NumPy: 负责数据计算
 - Matplotlib: 显示图片

In [6]:
%pip install ultralytics opencv-python numpy matplotlib

  Using cached ultralytics-8.4.128-py3-none-any.whl.metadata (45 kB)
  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached polars-1.44.0-py3-none-any.whl.metadata (11 kB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.1.6-py3-none-any.whl.metadata (13 kB)
  Using cached ultralytics_platform-0.1.14-py3-none-any.whl.metadata (8.4 kB)
  Using cached polars_runtime_32-1.44.0-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached ultralytics-8.4.128-py3-none-any.whl (1.4 MB)
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   -

### 2、安装YOLO v10
YOLO v10官方代码包含模型结构、训练以及推理相关文件。本实验通过下载官方的YOLOv10源码，并安装对应依赖库，主要步骤：
- 下载YOLOv10源码
- 安装YOLOv10依赖
- 检查YOLOv10是否可以正常调用

#### 2.1、下载YOLOv10源码

In [11]:
!git clone https://github.com/THU-MIG/yolov10.git

fatal: destination path 'yolov10' already exists and is not an empty directory.


#### 2.2、安装YOLO v10依赖

In [12]:
%cd yolov10

C:\\Users\\USER\Downloads\AI分层考\5-计算机视觉\yolov10


C:\\Users\\USER\AppData\Roaming\Python\Python312\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [16]:
!pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement torch==2.0.1 (from versions: 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0)
ERROR: No matching distribution found for torch==2.0.1


将YOLOv10 项目安装到当前Python环境，使其可以被NOtebook直接调用

In [17]:
!pip install -e .    # 表示安装当前目录下的包，并且在安装后可以直接修改源代码而不需要重新安装。

Defaulting to user installation because normal site-packages is not writeable


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


#### 2.3、检查YOLO V10是否可以正常调用

In [ ]:
from ultralytics import YOLO

print("ultralytics:", YOLO.__version__)

### 3、加载预训练模型

YOLOv10使用预训练权重进行目标检测
本实验使用官方提供的yolo10n.pt权重文件，因为yolov10n属于轻量模型，具有较快的推理速度，适合实时视频目标检测任务
- 下载模型权重
- 加载模型

#### 3.1、下载模型权重

In [ ]:
# 返回项目根目录
%cd ..

In [ ]:
# 下载 yolov10n.pt模型文件,也就是预训练权重文件
import urllib.request  #引入 urllib.request 模块为了下载文件

url = "https://github.com/THU-MIG/yolov10/releases/download/v1.0/yolov10n.pt"
save_path = "yolov10n.pt"
urllib.request.urlretrieve(url, save_path)
print(f"模型文件已下载并保存为: {save_path}")

#### 3.2、加载模型

In [ ]:
# 加载YOLOv10模型
model = YOLO("yolov10n.pt")  # 加载预训练模型
print("模型加载完成")


In [ ]:
# 下载 yolov10n.pt模型文件,也就是预训练权重文件
import urllib.request  #引入 urllib.request 模块为了下载文件

url = "https://github.com/THU-MIG/yolov10/releases/download/v1.0/yolov10n.pt"
save_path = "yolov10n.pt"
urllib.request.urlretrieve(url, save_path)
print(f"模型文件已下载并保存为: {save_path}")

### 4、输入测试视频

在进行视频检测前，需要准备输入视频文件

在YOLO v10项目中创建videos文件夹，并将测试视频test.mp4放入该文件夹中，本部分主要完成：

- 设置输入视频路径
- 检查视频文件是否存在
- 使用OpenCV读取视频基本信息

#### 4.1、设置输入视频路径

In [ ]:
# 设置视频输入和输出路径
video_path = "videos/test.mp4"  # 输入视频路径
print(f"输入视频路径: {video_path}")


#### 4.2、检查视频文件是否存在

In [ ]:
import os

if os.path.exists(video_path):
    print(f"视频文件存在: {video_path}")
else:
    print(f"视频文件不存在: {video_path}")

#### 4.3、使用OpenCV读取视频信息

OpenCV是计算机视觉领域常用的图像和视频处理库
这里使用OpenCV读取视频的基本信息，包括：
- 视频分辨率
- 视频帧率
- 视频总帧数

In [ ]:
# 导入必要的库
import cv2

cap = cv2.VideoCapture(video_path)  # 打开视频文件
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  # 获取视频宽度
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # 获取视频高度
fps = cap.get(cv2.CAP_PROP_FPS)  # 获取视频帧率
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # 获取视频总帧数
print(f"视频信息 - 宽度: {width}, 高度: {height}, 帧率: {fps}, 总帧数: {frame_count}")

### 5、执行目标检测
本部分将使用加载好的YOLOv10模型对视频进行目标检测
YOLOv10提供了predict接口用于模型推理
通过输入视频路径，模型可以自动完成视频读取、目标检测以及结果保存

In [ ]:
# 使用YOLOv10模型进行视频检测
# step1：读取视频第一帧
# step2：将第一帧输入YOLOv10模型进行检测，得到检测结果【输入NN】
# step3：将检测结果绘制在第一帧上，并显示出来【预测类别】
# step4：绘制边界框
# step5：保存结果

results = model.predict(
    source=video_path, 
    show=True, 
    save=True,   # 保存检测结果视频
    conf=0.25,   # 置信度阈值，是为了过滤掉低置信度的检测结果，默认值为0.25
    device=0
)  # device=0表示使用GPU，如果没有GPU可以设置为device='cpu'

print("检测完成，结果已保存。")

### 6、展示检测结果
YOLOv10默认保存的视频结果可能为AVI格式
由于部分浏览器的Jupyter Notebook对AVI格式兼容性较差，因此将检测结果转换为MP4格式

本部分主要完成：
- 安装视频转换依赖
- AVI转换MP4
- 展示最终检测视频

#### 6.1、安装视频转换依赖

In [ ]:
!pip install imageio-ffmpeg

#### 6.2 AVI转换MP4

使用imageio 读取YOLOv10生成的AVI视频文件
转换过程中：
- 保留原始视频帧
- 保留原始帧率
- 使用H.264编码生成MP4文件

转换后的MP4视频可以直接在NoteBook中播放

In [ ]:
import imageio.v2 as imageio

# YOLOv10模型检测视频的输出路径
avi_path = "runs/detect/predict/test.avi"  # YOLOv10模型检测视频的输出路径
mp4_path = "runs/detect/predict/result.mp4"  # YOLOv10模型检测视频的输出路径

# 读取AVI视频并转换为MP4格式
reader = imageio.get_reader(avi_path)
writer = imageio.get_writer(
    mp4_path, 
    fps=reader.get_meta_data()['fps'],
    codec='libx264',  # 使用H.264编码
)

# 将每一帧写入MP4视频
for frame in reader:
    writer.append_data(frame)

render.close() # 关闭AVI视频读取器
writer.close() # 关闭MP4视频写入器

print(f"视频已转换为MP4格式，保存路径: {mp4_path}")

SyntaxError: incomplete input (2345470699.py, line 12)

#### 6.4、展示最终检测视频

读取转换后的MP4文件，并在Notebook中显示目标检测结果
视频中包含YOLO v10预测的目标类别以及对应边界框

In [ ]:
from IPython.display import Video, display

# MP4检测结果路径
video_result_path = "runs/detect/predict/result.mp4"    

display(Video(video_result_path, embed=True))  # 在Jupyter Notebook中显示视频